In [ ]:
!pip install feedparser requests pandas beautifulsoup4 -q

import feedparser
import requests
import pandas as pd
import json
import re
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("СБОР 10 000 СТАТЕЙ НА КАЖДУЮ ИЗ 9 КАТЕГОРИЙ")
print("="*60)

# ============================================
# КАТЕГОРИИ KAGGLE (0-8)
# ============================================

kaggle_categories = {
    0: 'Общество/Россия',
    1: 'Экономика',
    2: 'Силовые структуры',
    3: 'Бывший СССР',
    4: 'Спорт',
    5: 'Забота о себе',
    6: 'Строительство',
    7: 'Туризм/Путешествия',
    8: 'Наука и техника'
}

# РАСШИРЕННЫЕ ключевые слова для каждой категории
category_keywords = {
    0: ['россия', 'общество', 'социальный', 'жители', 'граждане', 'москва', 'регион', 'россияне', 'государство', 'власть', 'президент', 'путин', 'правительство', 'дума'],
    1: ['экономика', 'бизнес', 'рубль', 'доллар', 'акции', 'инвестиции', 'нефть', 'газ', 'курс', 'биржа', 'цена', 'рынок', 'торги', 'санкции'],
    2: ['полиция', 'мвд', 'фсб', 'армия', 'военный', 'чп', 'пожар', 'авария', 'убийство', 'следствие', 'криминал', 'суд', 'задержание', 'терроризм'],
    3: ['украина', 'казахстан', 'беларусь', 'армения', 'грузия', 'молдова', 'снг', 'киев', 'минск', 'ташкент', 'баку', 'тбилиси', 'кишинев'],
    4: ['спорт', 'футбол', 'хоккей', 'чемпионат', 'матч', 'олимпиада', 'тренер', 'команда', 'гол', 'победа', 'турнир', 'клуб', 'игрок', 'лига'],
    5: ['здоровье', 'красота', 'фитнес', 'диета', 'медицина', 'уход', 'психология', 'спортзал', 'тренировка', 'витамины', 'лечение', 'врач', 'болезнь'],
    6: ['строительство', 'стройка', 'ремонт', 'жкх', 'дом', 'квартира', 'объект', 'строй', 'жилье', 'застройщик', 'новостройка', 'ремонт'],
    7: ['туризм', 'путешествие', 'отель', 'пляж', 'виза', 'билет', 'курорт', 'авиа', 'турция', 'египет', 'отели', 'отдых', 'поездка'],
    8: ['наука', 'технологии', 'космос', 'робот', 'исследование', 'открытие', 'инновации', 'мгу', 'академия', 'ученые', 'лаборатория', 'эксперимент']
}

# ============================================
# RSS ИСТОЧНИКИ
# ============================================

rss_sources = {
    'Lenta.ru': 'https://lenta.ru/rss',
    'Kommersant': 'https://www.kommersant.ru/RSS/news.xml',
    'AIF': 'https://aif.ru/rss/news',
    'Gazeta': 'https://www.gazeta.ru/export/rss/news.xml',
    'Izvestia': 'https://iz.ru/xml/rss.xml'
}

def parse_full_rss(feed_url, source_name):
    articles = []
    try:
        feed = feedparser.parse(feed_url)
        for entry in feed.entries:
            title = entry.get('title', '')
            full_text = entry.get('description', '')
            if not full_text or len(full_text) < 50:
                full_text = entry.get('summary', '')
            if not full_text or len(full_text) < 50:
                full_text = title

            if not title or len(title) < 10:
                continue

            combined = (title + ' ' + full_text).lower()

            best_cat = 0
            max_matches = 0
            for cat_id, keywords in category_keywords.items():
                matches = sum(1 for kw in keywords if kw in combined)
                if matches > max_matches:
                    max_matches = matches
                    best_cat = cat_id

            articles.append({
                'text': full_text,
                'category_id': best_cat
            })
        return articles
    except Exception as e:
        return []

# ============================================
# API LENTA.RU
# ============================================

class LentaParser:
    def _get_url(self, p):
        url = 'https://lenta.ru/search/v2/process?'\
            + 'from={}&'.format(p['from'])\
            + 'size={}&'.format(p['size'])\
            + 'sort={}&'.format(p['sort'])\
            + 'title_only={}&'.format(p['title_only'])\
            + 'domain={}&'.format(p['domain'])\
            + 'type={}&'.format(p['type'])\
            + 'modified%2Cformat=yyyy-MM-dd&'\
            + 'modified%2Cfrom={}&'.format(p['dateFrom'])\
            + 'modified%2Cto={}&'.format(p['dateTo'])\
            + 'query={}'.format(p['query'])
        return url

    def _get_table(self, p):
        url = self._get_url(p)
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        try:
            r = requests.get(url, headers=headers, timeout=10)
            return pd.DataFrame(r.json().get('matches', []))
        except:
            return pd.DataFrame()

    def get_articles(self, p, time_step=7):
        pc = p.copy()
        step = timedelta(days=time_step)
        d1 = datetime.strptime(pc['dateFrom'], '%Y-%m-%d')
        d2 = datetime.strptime(pc['dateTo'], '%Y-%m-%d')
        all_dfs = []
        while d1 <= d2:
            pc['dateTo'] = (d1 + step).strftime('%Y-%m-%d')
            if d1 + step > d2:
                pc['dateTo'] = d2.strftime('%Y-%m-%d')
            df = self._get_table(pc)
            if len(df) > 0:
                all_dfs.append(df)
            d1 += step + timedelta(days=1)
            pc['dateFrom'] = d1.strftime('%Y-%m-%d')
            time.sleep(0.2)
        return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

# ============================================
# СБОР ДАННЫХ
# ============================================

all_articles = []
TARGET_PER_CATEGORY = 10000  # ← 10 000 НА КАТЕГОРИЮ

print("\n1. СБОР ИЗ RSS ИСТОЧНИКОВ:")
for source_name, feed_url in rss_sources.items():
    print(f"  {source_name}...", end=' ')
    articles = parse_full_rss(feed_url, source_name)
    all_articles.extend(articles)
    print(f"{len(articles)} СТАТЕЙ")
    time.sleep(0.5)

print(f"\n  RSS НАКОПЛЕНО: {len(all_articles)} СТАТЕЙ")

# Сбор через API Lenta.ru для каждой категории
print("\n2. СБОР ИЗ LENTA.RU API:")
parser = LentaParser()
df_current = pd.DataFrame(all_articles) if all_articles else pd.DataFrame()

for cat_id, cat_name in kaggle_categories.items():
    current_count = len(df_current[df_current['category_id'] == cat_id]) if len(df_current) > 0 else 0

    if current_count >= TARGET_PER_CATEGORY:
        print(f"\n  {cat_name} (id={cat_id}): УЖЕ {current_count} (ЦЕЛЬ ДОСТИГНУТА)")
        continue

    print(f"\n  {cat_name} (id={cat_id}): НУЖНО {TARGET_PER_CATEGORY - current_count}")

    # Пробуем все ключевые слова по очереди
    for kw in category_keywords[cat_id][:5]:  # первые 5 ключевых слов
        if current_count >= TARGET_PER_CATEGORY:
            break

        params = {
            'query': kw,
            'from': '0',
            'size': '1000',
            'sort': '3',
            'title_only': '0',
            'domain': '1',
            'type': '1',
            'dateFrom': '2023-02-01',
            'dateTo': '2026-05-30'
        }

        df_api = parser.get_articles(params, time_step=7)

        for _, row in df_api.iterrows():
            if current_count >= TARGET_PER_CATEGORY:
                break
            title = row.get('title', '')
            text = row.get('text', '')
            full_text = title + ' ' + text if text else title
            if full_text and len(full_text) > 50:
                all_articles.append({
                    'text': full_text,
                    'category_id': cat_id
                })
                current_count += 1

        print(f"    {kw}: +{len(df_api)} (ВСЕГО: {current_count})")
        time.sleep(0.5)

# ============================================
# ОБРАБОТКА
# ============================================

df = pd.DataFrame(all_articles)
print(f"\nВСЕГО СОБРАНО: {len(df)} СТАТЕЙ")

df = df.drop_duplicates(subset=['text'])
print(f"ПОСЛЕ УДАЛЕНИЯ ДУБЛИКАТОВ: {len(df)}")

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^а-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nОЧИСТКА ТЕКСТА...")
df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 50]
print(f"ПОСЛЕ ОЧИСТКИ: {len(df)}")

# Балансировка до 10000
print("\nБАЛАНСИРОВКА ДАННЫХ (10000 НА КАТЕГОРИЮ):")
balanced_dfs = []
target = 10000

for cat_id in range(9):
    cat_df = df[df['category_id'] == cat_id]
    if len(cat_df) > target:
        cat_df = cat_df.sample(n=target, random_state=42)
    balanced_dfs.append(cat_df)
    print(f"  {cat_id} ({kaggle_categories[cat_id]}): {len(cat_df)}")

df_balanced = pd.concat(balanced_dfs, ignore_index=True)
print(f"\nИТОГО: {len(df_balanced)} СТАТЕЙ")

# Сохраняем
df_final = df_balanced[['clean_text', 'category_id']].copy()
df_final.columns = ['text', 'category_id']
df_final.to_csv('train_data_10k.csv', index=False, encoding='utf-8')
print("\n✅ СОХРАНЕНО: train_data_10k.csv")

with open('kaggle_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(kaggle_categories, f, ensure_ascii=False, indent=2)
print("✅ СОХРАНЕНО: kaggle_mapping.json")

print("\n" + "="*60)
print(f"ГОТОВО! {len(df_final)} СТАТЕЙ ПО 10000 НА КАТЕГОРИЮ")
print("="*60)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.8 MB/s eta 0:00:00
СБОР 10 000 СТАТЕЙ НА КАЖДУЮ ИЗ 9 КАТЕГОРИЙ

1. СБОР ИЗ RSS ИСТОЧНИКОВ:
  Lenta.ru... 200 СТАТЕЙ
  Kommersant... 431 СТАТЕЙ
  AIF... 300 СТАТЕЙ
  Gazeta... 0 СТАТЕЙ
  Izvestia... 0 СТАТЕЙ

  RSS НАКОПЛЕНО: 931 СТАТЕЙ

2. СБОР ИЗ LENTA.RU API:

  Общество/Россия (id=0): НУЖНО 9373
    россия: +151794 (ВСЕГО: 10000)

  Экономика (id=1): НУЖНО 9947
    экономика: +16012 (ВСЕГО: 10000)

  Силовые структуры (id=2): НУЖНО 9920
    полиция: +29338 (ВСЕГО: 10000)

  Бывший СССР (id=3): НУЖНО 9966
    украина: +145634 (ВСЕГО: 10000)

  Спорт (id=4): НУЖНО 9944
    спорт: +9342 (ВСЕГО: 9398)
    футбол: +3918 (ВСЕГО: 10000)

  Забота о себе (id=5): НУЖНО 9988
    здоровье: +18213 (ВСЕГО: 10000)

  Строительство (id=6): НУЖНО 9951
    строительство: +9781 (ВСЕГО: 9830)
    стройка: +882 (ВСЕГО: 10000)

  Туризм/Путешествия (id=7): НУЖНО 9988
    туризм: +2792 (ВСЕГО: 2804)
    пу

In [ ]:
from google.colab import files

# Скачиваем файл с данными для обучения
files.download('train_data_10k.csv')

# Скачиваем маппинг категорий
files.download('kaggle_mapping.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>